# 🏦 FinGuard AI — Bank Credit & Loan Default Risk Analytics Engine

**IBM SkillsBuild Data Analytics with AI — Academic Internship Program**  
**BharatCares & AICTE**  
**Student: Prashant**  

---

## Project Overview

FinGuard AI is an end-to-end credit risk analytics engine that:

- **Auto-detects** all columns, data types, and the target variable from the uploaded dataset (no hardcoded column names)
- Performs comprehensive **Exploratory Data Analysis (EDA)**
- Trains and evaluates **7 machine learning classifiers**
- Provides real-time **loan default risk scoring** with 5-tier risk classification
- Integrates with **IBM watsonx.ai** (Granite) for AI-powered risk narratives
- Exports a styled **multi-sheet Excel report**

**Dataset:** [Bank Credit Default — Loan Default Prediction (Kaggle)](https://www.kaggle.com/datasets/kornilovag94/bank-credit-default-loan-default)  
**Domain:** Finance & Banking — Credit Risk & Loan Default Prediction

---

## Table of Contents

1. Environment Setup & Imports
2. Data Loading
3. Auto-Detection Engine (Dataset Profiling)
4. Exploratory Data Analysis
5. Data Preprocessing
6. Model Training (7 Classifiers)
7. Model Evaluation & Comparison
8. Risk Prediction Engine
9. IBM watsonx.ai Integration
10. Excel Report Export
11. Streamlit App Launcher

## 1. Environment Setup & Imports

In [ ]:
# Install required packages (run once)
# !pip install streamlit pandas numpy scikit-learn xgboost plotly openpyxl ibm-watsonx-ai

In [ ]:
from __future__ import annotations

# ── Standard Library ───────────────────────────────────────────────────────
import os
import io
import time
import warnings
from dataclasses import dataclass, field
from typing import Any, Optional

warnings.filterwarnings('ignore')

# ── Data Processing ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Machine Learning ───────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    f1_score, accuracy_score
)
import xgboost as xgb

# ── Visualisation ──────────────────────────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Report Export ──────────────────────────────────────────────────────────
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

print('✅ All libraries loaded successfully.')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')
print(f'   sklearn  (check via sklearn.__version__)')
print(f'   xgboost {xgb.__version__}')

## 2. Data Loading

Upload the **Bank Credit Default — Loan Default Prediction** CSV from Kaggle:  
https://www.kaggle.com/datasets/kornilovag94/bank-credit-default-loan-default

> ⚠️ This notebook does **not** fabricate or generate data. It works exclusively with the actual uploaded dataset.

In [ ]:
# ── Load Dataset ────────────────────────────────────────────────────────────
# Update the path below to your downloaded Kaggle CSV file
DATASET_PATH = 'bank_credit_default.csv'   # ← change to your file name

def load_dataset(path: str) -> pd.DataFrame:
    """Load CSV or Excel dataset."""
    p = path.lower()
    if p.endswith('.csv'):
        return pd.read_csv(path)
    elif p.endswith(('.xlsx', '.xls')):
        return pd.read_excel(path)
    raise ValueError(f'Unsupported format: {path}')

df = load_dataset(DATASET_PATH)

print(f'✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   File: {DATASET_PATH}')
df.head()

## 3. Auto-Detection Engine (Dataset Profiling)

The engine automatically detects:
- Column names and data types
- Numeric / categorical / binary / datetime columns
- Target variable (using keyword heuristics)
- Identifier columns
- Missing values and duplicate rows
- Derivable features (DTI, LTV, utilisation)

**No column names are hardcoded anywhere.**

In [ ]:
@dataclass
class DataProfile:
    """Holds everything auto-discovered about the dataset."""
    df: pd.DataFrame
    n_rows: int = 0
    n_cols: int = 0
    all_columns: list = field(default_factory=list)
    numeric_cols: list = field(default_factory=list)
    categorical_cols: list = field(default_factory=list)
    binary_cols: list = field(default_factory=list)
    datetime_cols: list = field(default_factory=list)
    id_cols: list = field(default_factory=list)
    high_cardinality_cols: list = field(default_factory=list)
    target_col: Optional[str] = None
    target_classes: list = field(default_factory=list)
    target_distribution: dict = field(default_factory=dict)
    is_binary_target: bool = False
    missing_counts: dict = field(default_factory=dict)
    missing_pct: dict = field(default_factory=dict)
    total_missing: int = 0
    duplicate_rows: int = 0
    feature_cols: list = field(default_factory=list)
    numeric_features: list = field(default_factory=list)
    categorical_features: list = field(default_factory=list)
    derived_cols: dict = field(default_factory=dict)

In [ ]:
# ── Target Detection Keywords ───────────────────────────────────────────────
_TARGET_KEYWORDS = [
    'default', 'defaulted', 'loan_default', 'credit_default',
    'bad_loan', 'charged_off', 'status', 'target', 'label',
    'outcome', 'delinquent', 'repayment', 'paid', 'failed',
    'fraud', 'risk', 'approved',
]

_ID_KEYWORDS = ['id', 'uid', 'uuid', 'customer_id', 'loan_id',
                'account_id', 'applicant_id', 'index']


def _detect_target(df: pd.DataFrame) -> Optional[str]:
    """Heuristically detect the target column — no hardcoded names."""
    cols_lower = {c.lower().replace(' ', '_'): c for c in df.columns}
    for kw in _TARGET_KEYWORDS:
        if kw in cols_lower:
            return cols_lower[kw]
    for col_key, col_orig in cols_lower.items():
        for kw in _TARGET_KEYWORDS:
            if kw in col_key:
                return col_orig
    for col in df.select_dtypes(include=[np.number]).columns:
        if set(df[col].dropna().unique()).issubset({0, 1, 0.0, 1.0}):
            return col
    return None


def _detect_id_cols(df: pd.DataFrame) -> list:
    id_cols = []
    for col in df.columns:
        col_lower = col.lower().replace(' ', '_')
        if any(kw == col_lower or col_lower.endswith(f'_{kw}') or col_lower.startswith(f'{kw}_')
               for kw in _ID_KEYWORDS):
            id_cols.append(col)
    return list(dict.fromkeys(id_cols))


def _detect_derived(df: pd.DataFrame, numeric_cols: list) -> dict:
    derived = {}
    cols_lower = {c.lower().replace(' ', '_'): c for c in numeric_cols}
    debt = [c for k, c in cols_lower.items() if 'debt' in k or 'obligation' in k]
    income = [c for k, c in cols_lower.items() if 'income' in k or 'salary' in k]
    if debt and income:
        derived['debt_to_income_ratio'] = f'Calculated as {debt[0]} / {income[0]}'
    loan = [c for k, c in cols_lower.items() if 'loan' in k and ('amount' in k or 'value' in k)]
    prop = [c for k, c in cols_lower.items() if 'property' in k or 'collateral' in k]
    if loan and prop:
        derived['loan_to_value_ratio'] = f'Calculated as {loan[0]} / {prop[0]}'
    return derived


print('✅ Auto-detection functions defined.')

In [ ]:
def profile_dataset(df: pd.DataFrame) -> DataProfile:
    """Full auto-detection pipeline."""
    p = DataProfile(df=df)
    p.n_rows, p.n_cols = df.shape
    p.all_columns = list(df.columns)

    missing = df.isnull().sum()
    p.missing_counts = missing[missing > 0].to_dict()
    p.missing_pct = {k: round(v / p.n_rows * 100, 2) for k, v in p.missing_counts.items()}
    p.total_missing = int(missing.sum())
    p.duplicate_rows = int(df.duplicated().sum())

    for col in df.columns:
        dtype = df[col].dtype
        n_unique = df[col].nunique()
        if pd.api.types.is_datetime64_any_dtype(dtype):
            p.datetime_cols.append(col)
        elif pd.api.types.is_numeric_dtype(dtype):
            p.numeric_cols.append(col)
            if n_unique == 2:
                p.binary_cols.append(col)
        elif pd.api.types.is_object_dtype(dtype):
            p.categorical_cols.append(col)
            if n_unique == 2:
                p.binary_cols.append(col)
            if n_unique > 50:
                p.high_cardinality_cols.append(col)

    p.id_cols = _detect_id_cols(df)
    p.target_col = _detect_target(df)

    if p.target_col:
        t = df[p.target_col].dropna()
        p.target_classes = sorted(t.unique().tolist())
        p.target_distribution = t.value_counts().to_dict()
        p.is_binary_target = len(p.target_classes) == 2

    exclude = set(p.id_cols) | ({p.target_col} if p.target_col else set())
    p.feature_cols = [c for c in df.columns if c not in exclude]
    p.numeric_features = [c for c in p.feature_cols if c in p.numeric_cols]
    p.categorical_features = [c for c in p.feature_cols if c in p.categorical_cols]
    p.derived_cols = _detect_derived(df, p.numeric_features)
    return p


profile = profile_dataset(df)
print(f'✅ Dataset profiled.')
print(f'   Rows: {profile.n_rows:,}  |  Columns: {profile.n_cols}')
print(f'   Target detected: {profile.target_col}')
print(f'   Target classes: {profile.target_classes}')
print(f'   Binary target: {profile.is_binary_target}')
print(f'   Numeric features: {profile.numeric_features}')
print(f'   Categorical features: {profile.categorical_features}')
print(f'   ID columns: {profile.id_cols}')
print(f'   Missing values: {profile.total_missing:,}')
print(f'   Duplicate rows: {profile.duplicate_rows:,}')
if profile.derived_cols:
    print(f'   Derivable features: {list(profile.derived_cols.keys())}')

## 4. Exploratory Data Analysis

In [ ]:
# ── Summary Statistics ──────────────────────────────────────────────────────
print('=== SUMMARY STATISTICS (Numeric Features) ===')
if profile.numeric_features:
    stats = df[profile.numeric_features].describe().T
    stats['skewness'] = df[profile.numeric_features].skew()
    stats['kurtosis'] = df[profile.numeric_features].kurtosis()
    stats['missing'] = df[profile.numeric_features].isnull().sum()
    display(stats.round(4))
else:
    print('No numeric features detected.')

In [ ]:
# ── Target Distribution ─────────────────────────────────────────────────────
if profile.target_col:
    dist = profile.target_distribution
    total = sum(dist.values())
    labels = [str(k) for k in dist]
    values = list(dist.values())

    fig = make_subplots(rows=1, cols=2,
                        specs=[[{'type': 'bar'}, {'type': 'pie'}]],
                        subplot_titles=['Count per Class', 'Class Proportion'])
    fig.add_trace(go.Bar(x=labels, y=values,
                         marker_color=['#0f62fe', '#da1e28'],
                         text=[f'{v/total*100:.1f}%' for v in values],
                         textposition='outside'), row=1, col=1)
    fig.add_trace(go.Pie(labels=labels, values=values,
                         marker=dict(colors=['#0f62fe', '#da1e28']),
                         hole=0.4, textinfo='percent+label'), row=1, col=2)
    fig.update_layout(title=f'Target Distribution — {profile.target_col}',
                      showlegend=False, height=380)
    fig.show()

    # Imbalance analysis
    if profile.is_binary_target:
        vals_sorted = sorted(dist.values(), reverse=True)
        ratio = vals_sorted[0] / max(vals_sorted[1], 1)
        sev = 'Severe (>10:1)' if ratio > 10 else 'Moderate (3–10:1)' if ratio > 3 else 'Balanced (<3:1)'
        print(f'Class Imbalance Ratio: {ratio:.2f}:1  →  {sev}')
else:
    print('No target column detected.')

In [ ]:
# ── Numeric Feature Distributions ───────────────────────────────────────────
cols_to_plot = profile.numeric_features[:12]
if cols_to_plot:
    n_cols_grid = 3
    n_rows_grid = int(np.ceil(len(cols_to_plot) / n_cols_grid))
    palette = ['#0f62fe','#da1e28','#24a148','#8a3ffc','#ff832b','#1192e8',
               '#009d9a','#f1c21b','#d4bbff','#a7f0ba','#ffd6e8','#d0e2ff']

    fig = make_subplots(rows=n_rows_grid, cols=n_cols_grid,
                        subplot_titles=cols_to_plot,
                        vertical_spacing=0.1, horizontal_spacing=0.06)
    for i, col in enumerate(cols_to_plot):
        r, c = i // n_cols_grid + 1, i % n_cols_grid + 1
        fig.add_trace(go.Histogram(x=df[col].dropna(), nbinsx=30,
                                   marker_color=palette[i % len(palette)],
                                   name=col, showlegend=False), row=r, col=c)
    fig.update_layout(title='Numeric Feature Distributions', height=n_rows_grid * 220)
    fig.show()
else:
    print('No numeric features to plot.')

In [ ]:
# ── Correlation Heatmap ─────────────────────────────────────────────────────
if len(profile.numeric_features) >= 2:
    corr_cols = profile.numeric_features[:20]
    corr = df[corr_cols].corr(numeric_only=True)
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    z = corr.values.copy()
    z[mask] = None

    fig = go.Figure(go.Heatmap(
        z=z, x=corr_cols, y=corr_cols,
        colorscale=[[0, '#da1e28'], [0.5, '#ffffff'], [1.0, '#0f62fe']],
        zmin=-1, zmax=1,
        text=np.round(z, 2), texttemplate='%{text}',
        textfont=dict(size=9), colorbar=dict(title='r')
    ))
    fig.update_layout(title='Feature Correlation Matrix (Lower Triangle)',
                      height=max(400, len(corr_cols) * 30))
    fig.show()

    # Top correlations with target
    if profile.target_col and profile.target_col in profile.numeric_cols:
        corr_target = df[corr_cols + [profile.target_col]].corr()[profile.target_col]\
                        .drop(profile.target_col).sort_values(key=abs, ascending=False)
        print('\nTop correlations with target:')
        display(corr_target.head(10).to_frame())
else:
    print('Need ≥ 2 numeric features for correlation analysis.')

In [ ]:
# ── Box Plots by Target Class ───────────────────────────────────────────────
if profile.target_col and profile.numeric_features:
    cols_box = profile.numeric_features[:8]
    n_cols_g = 2
    n_rows_g = int(np.ceil(len(cols_box) / n_cols_g))
    palette = ['#0f62fe', '#da1e28']

    fig = make_subplots(rows=n_rows_g, cols=n_cols_g,
                        subplot_titles=cols_box, vertical_spacing=0.1)
    classes = [str(c) for c in profile.target_classes[:6]]
    for i, col in enumerate(cols_box):
        r, c = i // n_cols_g + 1, i % n_cols_g + 1
        for j, cls in enumerate(classes):
            subset = df[df[profile.target_col].astype(str) == cls][col].dropna()
            fig.add_trace(go.Box(y=subset, name=cls,
                                 marker_color=palette[j % len(palette)],
                                 legendgroup=cls, showlegend=(i == 0)), row=r, col=c)
    fig.update_layout(title=f'Numeric Features by Target Class ({profile.target_col})',
                      height=n_rows_g * 260)
    fig.show()

In [ ]:
# ── Missing Values Analysis ─────────────────────────────────────────────────
if profile.missing_pct:
    sorted_miss = sorted(profile.missing_pct.items(), key=lambda x: x[1], reverse=True)
    fig = go.Figure(go.Bar(
        x=[v for _, v in sorted_miss],
        y=[k for k, _ in sorted_miss],
        orientation='h',
        marker=dict(color=[v for _, v in sorted_miss],
                    colorscale=[[0,'#24a148'],[0.5,'#ff832b'],[1,'#da1e28']],
                    showscale=True),
        text=[f'{v:.1f}%' for _, v in sorted_miss],
        textposition='outside'
    ))
    fig.update_layout(title='Missing Values by Column (%)',
                      height=max(300, len(sorted_miss) * 28))
    fig.show()
else:
    print('✅ No missing values detected in the dataset!')

In [ ]:
# ── Outlier Analysis (IQR Method) ───────────────────────────────────────────
outlier_rows = []
for col in profile.numeric_features:
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((s < low) | (s > high)).sum())
    outlier_rows.append({
        'Feature': col, 'Q1': round(q1, 4), 'Q3': round(q3, 4),
        'IQR': round(iqr, 4), 'Lower Fence': round(low, 4),
        'Upper Fence': round(high, 4),
        'Outlier Count': n_out,
        'Outlier %': round(n_out / len(s) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_rows).sort_values('Outlier %', ascending=False)
print('Outlier Summary (IQR Method):')
display(outlier_df)
print(f'Total outlier instances: {outlier_df["Outlier Count"].sum():,}')

## 5. Data Preprocessing

In [ ]:
def preprocess_for_model(df: pd.DataFrame, profile: DataProfile):
    """
    Safe preprocessing pipeline:
    - Drops ID and datetime columns
    - Label-encodes target
    - Fills numeric NaNs with median
    - Fills categorical NaNs with mode
    - One-hot encodes low-cardinality categoricals (≤20 unique)
    - Drops high-cardinality categoricals (>20 unique)
    """
    if not profile.target_col:
        raise ValueError('No target column detected.')

    work = df.copy()
    drop_cols = list(set(profile.id_cols) | set(profile.datetime_cols))
    work.drop(columns=[c for c in drop_cols if c in work.columns], inplace=True)

    y_raw = work.pop(profile.target_col)
    le = LabelEncoder()
    y = le.fit_transform(y_raw.astype(str))
    target_classes = le.classes_.tolist()

    for col in work.select_dtypes(include=[np.number]).columns:
        work[col] = work[col].fillna(work[col].median())

    cat_cols = work.select_dtypes(include=['object', 'category']).columns.tolist()
    for col in cat_cols:
        fill_val = work[col].mode()[0] if not work[col].mode().empty else 'Unknown'
        work[col] = work[col].fillna(fill_val)

    low_card = [c for c in cat_cols if work[c].nunique() <= 20]
    high_card = [c for c in cat_cols if work[c].nunique() > 20]
    if high_card:
        print(f'  ⚠ Dropping high-cardinality columns: {high_card}')
    work.drop(columns=high_card, inplace=True)
    if low_card:
        work = pd.get_dummies(work, columns=low_card, drop_first=True)

    bool_cols = work.select_dtypes(include=['bool']).columns
    work[bool_cols] = work[bool_cols].astype(int)

    return work.values, y, list(work.columns), target_classes


X, y, feature_names, target_classes = preprocess_for_model(df, profile)

print(f'✅ Preprocessing complete.')
print(f'   X shape: {X.shape}')
print(f'   y shape: {y.shape}')
print(f'   Features ({len(feature_names)}): {feature_names}')
print(f'   Target classes: {target_classes}')
print(f'   Class distribution in y: {dict(zip(*np.unique(y, return_counts=True)))}')

In [ ]:
# ── Train / Test Split & Feature Scaling ────────────────────────────────────
TEST_SIZE = 0.20
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# XGBoost scale_pos_weight for imbalanced datasets
n_neg = np.sum(y_train == 0)
n_pos = np.sum(y_train == 1)
spw = n_neg / max(n_pos, 1)

print(f'Train: {X_train_sc.shape[0]:,}  |  Test: {X_test_sc.shape[0]:,}')
print(f'XGBoost scale_pos_weight: {spw:.2f}')

## 6. Model Training (7 Classifiers)

In [ ]:
# ── Model Registry ──────────────────────────────────────────────────────────
models = {
    'Logistic Regression':   LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Decision Tree':         DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42),
    'Random Forest':         RandomForestClassifier(n_estimators=200, max_depth=10,
                                                    class_weight='balanced', random_state=42, n_jobs=-1),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42),
    'XGBoost':               xgb.XGBClassifier(n_estimators=200, max_depth=6, eval_metric='logloss',
                                               scale_pos_weight=spw, random_state=42),
    'K-Nearest Neighbours':  KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
    'Naive Bayes':           GaussianNB(),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    print(f'Training {name}...', end=' ')
    t0 = time.time()
    model.fit(X_train_sc, y_train)
    elapsed = time.time() - t0

    y_pred = model.predict(X_test_sc)
    y_proba = model.predict_proba(X_test_sc)[:, 1] if hasattr(model, 'predict_proba') else None

    roc = roc_auc_score(y_test, y_proba) if y_proba is not None and len(np.unique(y_test)) == 2 else 0.5
    avg_p = average_precision_score(y_test, y_proba) if y_proba is not None else 0
    cv_scores = cross_val_score(model, X_train_sc, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

    rep = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    results[name] = {
        'model': model,
        'accuracy':      accuracy_score(y_test, y_pred),
        'precision':     rep.get('weighted avg', {}).get('precision', 0),
        'recall':        rep.get('weighted avg', {}).get('recall', 0),
        'f1':            rep.get('weighted avg', {}).get('f1-score', 0),
        'roc_auc':       roc,
        'avg_precision': avg_p,
        'cv_mean':       cv_scores.mean(),
        'cv_std':        cv_scores.std(),
        'train_time':    elapsed,
        'y_pred':        y_pred,
        'y_proba':       y_proba,
        'confusion':     confusion_matrix(y_test, y_pred),
        'fi':            getattr(model, 'feature_importances_', None)
                         if not hasattr(model, 'feature_importances_')
                         else model.feature_importances_,
    }
    print(f'✓  {elapsed:.2f}s  |  ROC-AUC: {roc:.4f}  |  F1: {results[name]["f1"]:.4f}')

best_name = max(results, key=lambda n: results[n]['roc_auc'])
print(f'\n🏆 Best model: {best_name}  (ROC-AUC: {results[best_name]["roc_auc"]:.4f})')

## 7. Model Evaluation & Comparison

In [ ]:
# ── Metrics Summary Table ───────────────────────────────────────────────────
summary_rows = []
for name, res in results.items():
    summary_rows.append({
        'Model':            f'🏆 {name}' if name == best_name else name,
        'Accuracy':         round(res['accuracy'], 4),
        'Precision':        round(res['precision'], 4),
        'Recall':           round(res['recall'], 4),
        'F1-Score':         round(res['f1'], 4),
        'ROC-AUC':          round(res['roc_auc'], 4),
        'Avg Precision':    round(res['avg_precision'], 4),
        'CV AUC (mean±std)': f"{res['cv_mean']:.4f} ± {res['cv_std']:.4f}",
        'Train Time (s)':   round(res['train_time'], 3),
    })

metrics_df = pd.DataFrame(summary_rows)
display(metrics_df.style.highlight_max(subset=['Accuracy','F1-Score','ROC-AUC'],
                                        color='#defbe6').format(precision=4))

In [ ]:
# ── ROC Curves ─────────────────────────────────────────────────────────────
if profile.is_binary_target:
    palette = ['#0f62fe','#da1e28','#24a148','#8a3ffc','#ff832b','#1192e8','#009d9a']
    fig = go.Figure()
    fig.add_shape(type='line', x0=0, y0=0, x1=1, y1=1,
                  line=dict(dash='dot', color='grey', width=1))
    for i, (name, res) in enumerate(results.items()):
        if res['y_proba'] is not None:
            fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
            fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                                     name=f"{name} (AUC={res['roc_auc']:.3f})",
                                     line=dict(color=palette[i % len(palette)], width=2)))
    fig.update_layout(title='ROC Curves — All Models',
                      xaxis_title='False Positive Rate',
                      yaxis_title='True Positive Rate', height=480)
    fig.show()

In [ ]:
# ── Confusion Matrix (Best Model) ───────────────────────────────────────────
best_res = results[best_name]
cm = best_res['confusion']
labels = [str(c) for c in target_classes]

fig = go.Figure(go.Heatmap(
    z=cm, x=[f'Pred: {l}' for l in labels], y=[f'True: {l}' for l in labels],
    colorscale=[[0,'#ffffff'],[1,'#0f62fe']],
    text=[[str(v) for v in row] for row in cm],
    texttemplate='%{text}', textfont=dict(size=18), showscale=False
))
fig.update_layout(title=f'Confusion Matrix — {best_name}', height=350)
fig.show()

if cm.size == 4:
    tn, fp, fn, tp = cm.ravel()
    print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')
    print(f'Sensitivity: {tp/(tp+fn+1e-9):.4f}  |  Specificity: {tn/(tn+fp+1e-9):.4f}')

In [ ]:
# ── Feature Importance (Best Model) ────────────────────────────────────────
fi = best_res.get('fi')
if fi is not None:
    names_arr = np.array(feature_names)
    idx = np.argsort(fi)[-20:]
    fig = go.Figure(go.Bar(
        x=fi[idx], y=names_arr[idx], orientation='h',
        marker=dict(color=fi[idx],
                    colorscale=[[0,'#e5e7eb'],[1,'#0f62fe']]),
    ))
    fig.update_layout(title=f'Top 20 Feature Importances — {best_name}', height=500)
    fig.show()
else:
    print(f'{best_name} does not expose feature importances.')

In [ ]:
# ── Full Classification Report (Best Model) ─────────────────────────────────
print(f'=== Classification Report — {best_name} ===')
print(classification_report(y_test, best_res['y_pred'],
                             target_names=[str(c) for c in target_classes],
                             zero_division=0))

## 8. Risk Prediction Engine

Five-tier risk classification based on default probability.

In [ ]:
def compute_risk_score(proba: float) -> tuple:
    """Convert default probability to risk tier."""
    if proba >= 0.75:   return '🔴 Very High Risk', '#da1e28'
    elif proba >= 0.55: return '🟠 High Risk',      '#ff832b'
    elif proba >= 0.35: return '🟡 Moderate Risk',  '#f1c21b'
    elif proba >= 0.15: return '🟢 Low Risk',        '#24a148'
    else:               return '✅ Very Low Risk',   '#198038'


def predict_applicant(model, scaler, input_dict: dict, feature_names: list):
    """Predict default risk for a single applicant."""
    row = pd.DataFrame([input_dict])
    for col in feature_names:
        if col not in row.columns:
            row[col] = 0
    row = row[feature_names].fillna(0)
    X_in = scaler.transform(row.values.astype(float))
    pred = int(model.predict(X_in)[0])
    proba = float(model.predict_proba(X_in)[0][1]) if hasattr(model, 'predict_proba') else float(pred)
    label, colour = compute_risk_score(proba)
    return pred, proba, label


# ── Demo prediction using median values from the dataset ───────────────────
sample_input = {col: float(df[col].median()) for col in feature_names if col in df.columns}
# Fill any OHE columns with 0
for col in feature_names:
    if col not in sample_input:
        sample_input[col] = 0

pred_class, pred_proba, risk_label = predict_applicant(
    best_res['model'], scaler, sample_input, feature_names
)

print('=== Sample Prediction (Median-Value Applicant) ===')
print(f'Predicted Class:      {pred_class}  ({target_classes[pred_class] if pred_class < len(target_classes) else pred_class})')
print(f'Default Probability:  {pred_proba:.1%}')
print(f'Risk Tier:            {risk_label}')

## 9. IBM watsonx.ai Integration

AI-powered risk narratives and insights using IBM Granite foundation models.  
Requires IBM Cloud credentials — leave blank to skip.

**Model used:** `ibm/granite-13b-instruct-v2`

In [ ]:
# ── IBM watsonx.ai Config ───────────────────────────────────────────────────
WATSONX_API_KEY    = ''   # ← your IBM Cloud API key
WATSONX_PROJECT_ID = ''   # ← your watsonx.ai project ID
WATSONX_URL        = 'https://us-south.ml.cloud.ibm.com'


def get_watsonx_model(api_key: str, project_id: str, url: str):
    from ibm_watsonx_ai import APIClient, Credentials  # type: ignore[import-untyped]
    from ibm_watsonx_ai.foundation_models import ModelInference  # type: ignore[import-untyped]
    credentials = Credentials(url=url, api_key=api_key)
    client = APIClient(credentials)
    return ModelInference(
        model_id='ibm/granite-13b-instruct-v2',
        api_client=client,
        project_id=project_id,
        params={'max_new_tokens': 600, 'decoding_method': 'greedy', 'repetition_penalty': 1.1}
    )


wx_model = None
if WATSONX_API_KEY and WATSONX_PROJECT_ID:
    try:
        wx_model = get_watsonx_model(WATSONX_API_KEY, WATSONX_PROJECT_ID, WATSONX_URL)
        print('✅ Connected to IBM watsonx.ai (Granite)')
    except Exception as e:
        print(f'⚠ watsonx.ai connection failed: {e}')
else:
    print('ℹ watsonx.ai credentials not set — AI narrative skipped.')

In [ ]:
# ── Generate AI Risk Explanation ─────────────────────────────────────────────
if wx_model is not None:
    fi_arr = best_res.get('fi')
    top_features_text = ''
    if fi_arr is not None:
        idx_top = np.argsort(fi_arr)[::-1][:8]
        top_features_text = '\n'.join(
            f'  - {feature_names[i]}: {sample_input.get(feature_names[i], 0):.4f}'
            for i in idx_top
        )

    prompt = f"""You are FinGuard AI, an expert credit risk analyst.
Analyze the following loan applicant risk profile and provide a structured assessment.

PREDICTION: {'DEFAULT' if pred_class == 1 else 'NON-DEFAULT'}
DEFAULT PROBABILITY: {pred_proba:.1%}
RISK TIER: {risk_label}

TOP CONTRIBUTING FEATURES:
{top_features_text}

Provide:
1. Executive Summary (2-3 sentences)
2. Key Risk Drivers (bullet points)
3. Recommended Decision (Approve / Conditional / Reject)
4. Risk Mitigation Actions
Response:"""

    response = wx_model.generate_text(prompt=prompt)
    print('=== IBM watsonx.ai — AI Risk Narrative ===')
    print(response)
else:
    print('ℹ Skipping AI narrative (watsonx.ai not connected).')

## 10. Excel Report Export

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from datetime import datetime


def _header_fill(c='0F62FE'):
    return PatternFill(start_color=c, end_color=c, fill_type='solid')

def _border():
    t = Side(style='thin', color='D0D0D0')
    return Border(left=t, right=t, top=t, bottom=t)

def _write_df(ws, df_in, start_row=1, colour='0F62FE'):
    for ci, col_name in enumerate(df_in.columns, 1):
        c = ws.cell(row=start_row, column=ci, value=str(col_name))
        c.fill = _header_fill(colour)
        c.font = Font(bold=True, color='FFFFFF', name='Calibri', size=11)
        c.alignment = Alignment(horizontal='center')
        c.border = _border()
    for ri, row in enumerate(df_in.itertuples(index=False), start_row + 1):
        bg = 'F2F4FF' if ri % 2 == 0 else 'FFFFFF'
        for ci, val in enumerate(row, 1):
            cell = ws.cell(row=ri, column=ci)
            cell.value = round(float(val), 4) if isinstance(val, float) else val
            cell.fill = PatternFill(start_color=bg, end_color=bg, fill_type='solid')
            cell.font = Font(name='Calibri', size=10)
            cell.border = _border()
    for col in ws.iter_cols(min_row=start_row, max_row=ws.max_row):
        ws.column_dimensions[col[0].column_letter].width = \
            min(max(len(str(c.value or '')) for c in col) + 4, 32)


wb = Workbook()
wb.remove(wb.active)
ts = datetime.now().strftime('%Y-%m-%d %H:%M')

# Sheet 1: Executive Summary
ws1 = wb.create_sheet('Executive Summary')
ws1.merge_cells('A1:F1')
ws1['A1'] = 'FinGuard AI — Executive Summary'
ws1['A1'].font = Font(bold=True, size=16, color='0F62FE', name='Calibri')
ws1['A1'].alignment = Alignment(horizontal='center')
ws1['A1'].fill = PatternFill(start_color='EEF2FF', end_color='EEF2FF', fill_type='solid')
ws1.row_dimensions[1].height = 30
summary_data = [
    ('Project', 'FinGuard AI'),
    ('Student', 'Prashant'),
    ('Program', 'IBM SkillsBuild Data Analytics with AI | BharatCares & AICTE'),
    ('Domain', 'Finance & Banking — Credit Risk'),
    ('Generated', ts),
    ('Dataset Rows', f'{profile.n_rows:,}'),
    ('Dataset Columns', str(profile.n_cols)),
    ('Target Variable', str(profile.target_col)),
    ('Best Model', best_name),
    ('Best ROC-AUC', f"{results[best_name]['roc_auc']*100:.2f}%"),
    ('Best F1-Score', f"{results[best_name]['f1']*100:.2f}%"),
    ('Models Evaluated', str(len(results))),
]
for i, (k, v) in enumerate(summary_data, 3):
    ws1.cell(row=i, column=2, value=k).font = Font(bold=True, name='Calibri', size=11)
    ws1.cell(row=i, column=3, value=v).font = Font(name='Calibri', size=11)
ws1.column_dimensions['B'].width = 28
ws1.column_dimensions['C'].width = 55

# Sheet 2: Model Results
ws2 = wb.create_sheet('Model Results')
mr_df = pd.DataFrame([
    {'Model': n, 'Accuracy': round(r['accuracy']*100,2),
     'Precision': round(r['precision']*100,2), 'Recall': round(r['recall']*100,2),
     'F1-Score': round(r['f1']*100,2), 'ROC-AUC': round(r['roc_auc']*100,2),
     'CV Mean AUC': round(r['cv_mean']*100,2), 'CV Std': round(r['cv_std'],4),
     'Train Time(s)': round(r['train_time'],3)}
    for n, r in results.items()
])
_write_df(ws2, mr_df, start_row=1, colour='24A148')

# Sheet 3: Feature Importance
ws3 = wb.create_sheet('Feature Importance')
fi_arr = best_res.get('fi')
if fi_arr is not None:
    fi_df = pd.DataFrame({
        'Rank': range(1, len(fi_arr)+1),
        'Feature': np.array(feature_names)[np.argsort(fi_arr)[::-1]],
        'Importance': np.sort(fi_arr)[::-1].round(6)
    })
    _write_df(ws3, fi_df, start_row=1, colour='8A3FFC')

# Sheet 4: Data Profile
ws4 = wb.create_sheet('Data Profile')
dp_rows = []
for col in profile.all_columns:
    col_type = ('Target' if col == profile.target_col else
                'ID' if col in profile.id_cols else
                'Numeric' if col in profile.numeric_cols else 'Categorical')
    dp_rows.append({'Column': col, 'Type': col_type, 'Dtype': str(df[col].dtype),
                    'Unique': df[col].nunique(),
                    'Missing %': round(profile.missing_pct.get(col, 0), 2)})
_write_df(ws4, pd.DataFrame(dp_rows), start_row=1, colour='FF832B')

REPORT_FILE = 'FinGuard_AI_Report.xlsx'
wb.save(REPORT_FILE)
print(f'✅ Excel report saved: {REPORT_FILE}')

## 11. Launch Streamlit App

The full interactive Streamlit application is in `app.py` and `utils/`.  
Run the cell below to launch it.

In [ ]:
# Launch FinGuard AI Streamlit application
# Run this cell or execute in your terminal:
#     python -m streamlit run app.py

import subprocess, sys
print('Starting FinGuard AI...')
print('Open your browser at: http://localhost:8501')
# subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'app.py'])
# ↑ Uncomment the line above to auto-launch, or run in terminal instead

---

## Summary

| Step | Deliverable |
|------|-------------|
| Data Loading | Loaded actual Kaggle dataset — no fabricated data |
| Auto-Detection | All columns, types, target, IDs detected dynamically |
| EDA | Distributions, correlations, box plots, outliers, class balance |
| Preprocessing | Median imputation, OHE, feature scaling |
| Model Training | 7 classifiers trained and cross-validated |
| Evaluation | ROC-AUC, F1, confusion matrix, feature importance |
| Risk Prediction | 5-tier real-time scoring engine |
| IBM watsonx.ai | Granite LLM risk narrative generation |
| Excel Report | Multi-sheet styled report exported |
| Streamlit App | Full interactive deployment-ready application |

---

**IBM SkillsBuild Data Analytics with AI · BharatCares & AICTE · Prashant**